# Discovering Reddit communities: where data scientists go on Reddit
02807 Computational Tools for Data Science, Group 31

**How to run:** `pip install numpy pandas scipy scikit-learn matplotlib networkx requests`, then run
`python fetch_data.py` once (1-2 hours; every API answer is cached in `data/cache`, so an
interrupted run continues where it stopped). After that, run this notebook top to bottom
(a few minutes). All shared functions are in `helper_functions.py`.

Pipeline: a random sample of 3,000 users of four data science subreddits -> the subreddits they are active in -> each
subreddit as a set of users -> MinHash + LSH (Week 4) -> similarity graph -> Louvain (Week 7)
-> community labels with TF.IDF (Week 1) and the log-odds ratio with informative Dirichlet
prior (outside method). Evaluation: exact Jaccard of all pairs, null model, hold-out
recommendation.

In [1]:
import time
from itertools import combinations
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

import helper_functions as hp

SEEDS = ["datascience", "MachineLearning", "dataengineering", "statistics"]  # contain their users by construction, so not in the graph
MIN_COUNT = 2          # a user belongs to a subreddit with at least this many posts/comments there
MIN_USERS = 10         # subreddits with fewer of our users are dropped
THRESHOLD = 0.10       # Jaccard threshold for an edge: the top <1% of pairs, keeps the graph connected
N_HASH = 300           # signature length; longer than the typical 100 because t is low
BANDS, ROWS = 150, 2   # LSH banding, N_HASH = BANDS * ROWS; S-curve threshold (1/b)^(1/r) = 0.08 just below t
N_COLOURED = 8         # the largest communities get a colour in the map and a row in the label table
FIGURES = Path("report/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

In [2]:
# Figure style: Times-like serif font as in the report, thin marks, light solid grid
INK, INK2, GRID, GREY = "#0b0b0b", "#52514e", "#e4e3df", "#c4c3bd"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COL = 3.4  # width of one report column in inches
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "lines.linewidth": 1.4, "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})

## 1. Data: one set of users per subreddit

In [3]:
activity = pd.read_csv("data/user_activity_2026-06_09.csv.gz")
comments = pd.read_csv("data/comment_samples_2026-09.csv.gz")

sets = hp.subreddit_sets(activity, min_count=MIN_COUNT, min_users=MIN_USERS, exclude=SEEDS)
names, n_sets, n_users = sets["names"], len(sets["names"]), sets["n_users"]
user_sets = [set(sets["user_idx"][sets["set_idx"] == s]) for s in range(n_sets)]
size = pd.Series(sets["sizes"], index=names)

data_summary = pd.Series({
    "users": activity["user"].nunique(),
    "(user, subreddit) rows": len(activity),
    "subreddits": activity["subreddit"].nunique(),
    "median subreddits per user": activity.groupby("user").size().median(),
    f"rows with count >= {MIN_COUNT}": int((activity["count"] >= MIN_COUNT).sum()),
    f"subreddits with >= {MIN_USERS} users (kept)": n_sets,
    "users in kept sets": n_users,
    "memberships (set elements)": len(sets["set_idx"]),
    "pairs of subreddits": n_sets * (n_sets - 1) // 2,
    "comments for labels": len(comments),
})
print(data_summary.to_string())
print(size.describe().round(0).to_string())

users                                   2999.0
(user, subreddit) rows                 91836.0
subreddits                             19626.0
median subreddits per user                19.0
rows with count >= 2                   57901.0
subreddits with >= 10 users (kept)       956.0
users in kept sets                      2470.0
memberships (set elements)             27411.0
pairs of subreddits                   456490.0
comments for labels                    64063.0
count    956.0
mean      29.0
std       36.0
min       10.0
25%       12.0
50%       17.0
75%       30.0
max      505.0


## 2. Exact Jaccard similarity of all pairs (reference for LSH)

In [4]:
start = time.perf_counter()
J_naive = hp.all_pairs_jaccard(user_sets)
time_naive = time.perf_counter() - start

J = hp.jaccard_matrix(sets["set_idx"], sets["user_idx"], n_sets)
i_up, j_up = np.triu_indices(n_sets, k=1)
upper = J[i_up, j_up]
print(f"all-pairs Jaccard with Python sets: {time_naive:.2f} s for {len(J_naive):,} pairs")
print("pairs with J = 0:", round((upper == 0).mean(), 3))
print("quantiles of J (50, 90, 99, 99.9%):", np.quantile(upper, [0.5, 0.9, 0.99, 0.999]).round(4))
print(f"pairs with J >= {THRESHOLD}: {(upper >= THRESHOLD).sum()} ({100 * (upper >= THRESHOLD).mean():.1f}%)")

all-pairs Jaccard with Python sets: 0.41 s for 456,490 pairs
pairs with J = 0: 0.674
quantiles of J (50, 90, 99, 99.9%): [0.     0.0333 0.0849 0.1799]
pairs with J >= 0.1: 2919 (0.6%)


## 3. MinHash + LSH

In [5]:
start = time.perf_counter()
sig = hp.minhash(sets["set_idx"], sets["user_idx"], n_sets, k=N_HASH)
candidates = hp.lsh_pairs(sig, BANDS, ROWS)
similar = hp.verify(candidates, user_sets, THRESHOLD)
time_lsh = time.perf_counter() - start

quality = hp.lsh_quality(candidates, J, THRESHOLD)
print(pd.Series(quality).round(3).to_string())
print(f"time: MinHash + LSH + verification {time_lsh:.2f} s, all pairs {time_naive:.2f} s")
print(f"threshold (1/b)^(1/r) = {(1 / BANDS) ** (1 / ROWS):.3f}")

pairs         456490.000
candidates     21677.000
similar         2919.000
false_neg        267.000
false_pos      19025.000
recall             0.909
time: MinHash + LSH + verification 0.16 s, all pairs 0.41 s
threshold (1/b)^(1/r) = 0.082


In [6]:
# Other choices of n, b and r (incl. the lecture example n=100, b=20, r=5), evaluated against the exact pairs
rows = []
for n_hash, b, r in [(100, 20, 5), (100, 50, 2), (N_HASH, N_HASH, 1), (N_HASH, BANDS, ROWS), (N_HASH, N_HASH // 3, 3)]:
    signature = sig if n_hash == N_HASH else hp.minhash(sets["set_idx"], sets["user_idx"], n_sets, k=n_hash)
    cands = hp.lsh_pairs(signature, b, r)
    rows.append({"n": n_hash, "b": b, "r": r, "threshold": (1 / b) ** (1 / r), **hp.lsh_quality(cands, J, THRESHOLD)})
band_choices = pd.DataFrame(rows)
print(band_choices.round(3).to_string(index=False))

  n   b  r  threshold  pairs  candidates  similar  false_neg  false_pos  recall
100  20  5      0.549 456490          23     2919       2897          1   0.008
100  50  2      0.141 456490        9117     2919       1131       7329   0.613
300 300  1      0.003 456490      146552     2919          0     143633   1.000
300 150  2      0.082 456490       21677     2919        267      19025   0.909
300 100  3      0.215 456490        1358     2919       2184        623   0.252


In [7]:
# Runtime as the number of subreddits grows (random subsets)
scaling = hp.scaling_experiment(sets, [n_sets // 8, n_sets // 4, n_sets // 2, n_sets], N_HASH, BANDS, ROWS, THRESHOLD)
print(scaling.round(3).to_string(index=False))

 subreddits  pairs  time exact [s]  time LSH [s]  recall
        119   7021           0.006         0.011   0.896
        239  28441           0.016         0.020   0.925
        478 114003           0.065         0.044   0.920
        956 456490           0.322         0.111   0.909


In [8]:
# Figure 3: (a) S-curves and observed candidates, (b) runtime as the data grows
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(COL * 2.05, 2.1))
s = np.linspace(0, 0.45, 400)
for (b, r), color, note in [((N_HASH, 1), SERIES[1], ""), ((BANDS, ROWS), SERIES[0], ", ours"), ((20, 5), SERIES[2], ", lecture")]:
    ax1.plot(s, hp.s_curve(s, b, r), color=color, label=f"$b={b},\\ r={r}${note}")
is_candidate = np.array([(a, c) in candidates for a, c in zip(i_up, j_up)])
bins = np.r_[0, np.linspace(0.025, 0.45, 18)]
observed = pd.DataFrame({"bin": np.digitize(upper, bins) - 1, "J": upper, "cand": is_candidate}).groupby("bin").agg(
    J=("J", "mean"), share=("cand", "mean"), pairs=("cand", "size"))
observed = observed[observed["pairs"] >= 5]
ax1.plot(observed["J"], observed["share"], "o", ms=3.5, color=SERIES[0], mec="white", mew=0.6,
         label=f"observed ($b={BANDS}, r={ROWS}$)")
ax1.axvline(THRESHOLD, color=INK2, lw=0.8)
ax1.text(THRESHOLD + 0.005, 0.04, f"$t={THRESHOLD}$", color=INK2, fontsize=7)
ax1.set(xlabel="Jaccard similarity $J$ of a pair", ylabel="P(pair becomes candidate)", xlim=(0, 0.45), ylim=(-0.02, 1.02))
ax1.legend(loc="center right")
ax1.set_title("(a) S-curves and observed candidates", loc="left")
ax2.plot(scaling["pairs"], scaling["time exact [s]"], "o-", ms=3.5, color=SERIES[1], mec="white", mew=0.6,
         label="all pairs, exact")
ax2.plot(scaling["pairs"], scaling["time LSH [s]"], "o-", ms=3.5, color=SERIES[0], mec="white", mew=0.6,
         label="MinHash + LSH + verification")
ax2.set(xscale="log", yscale="log", xlabel="number of subreddit pairs", ylabel="runtime [s]")
ax2.legend(loc="upper left")
ax2.set_title("(b) Runtime as the data grows", loc="left")
fig.tight_layout(w_pad=1.5)
fig.savefig(FIGURES / "fig_lsh.pdf")

## 4. Similarity graph and Louvain

In [9]:
adj = hp.similarity_graph(similar, names)
levels = hp.louvain(adj)
for level, (membership, q) in enumerate(levels):
    print(f"level {level}: {len(set(membership.values()))} communities, Q = {q:.3f}")
membership, q_final = levels[-1]

community_ids = pd.Series(membership).value_counts().index          # largest community first
sub_comm = {sub: int(np.flatnonzero(community_ids == c)[0]) for sub, c in membership.items()}
n_comm = len(community_ids)
print("subreddits in graph:", len(adj), "of", n_sets, "| edges:", len(similar))

# subreddits ranked inside each community by their weighted degree to the same community
strength = {sub: sum(w for nb, w in adj[sub].items() if sub_comm[nb] == sub_comm[sub]) for sub in adj}
communities = (pd.DataFrame({"subreddit": list(sub_comm), "community": list(sub_comm.values())})
               .assign(strength=lambda t: t["subreddit"].map(strength), users=lambda t: t["subreddit"].map(size))
               .sort_values(["community", "strength"], ascending=[True, False]))
members_of = communities.groupby("community")["subreddit"].apply(list)
community_table = communities.groupby("community").agg(
    size=("subreddit", "size"), top_subreddits=("subreddit", lambda s: ", ".join(s[:6])))
print(community_table.to_string())

level 0: 140 communities, Q = 0.733
level 1: 62 communities, Q = 0.770
level 2: 47 communities, Q = 0.773
subreddits in graph: 831 of 956 | edges: 2652
           size                                                                                 top_subreddits
community                                                                                                     
0            94             justincaseyoumissedit, SpaceXBets, USNEWS, anticapitalism, NewsThread, remoteworks
1            83               TikTokCringe, SipsTea, mildlyinteresting, nextfuckinglevel, PoursTea, technology
2            78                                    agi, Anthropic, accelerate, singularity, OpenAI, ClaudeCode
3            77  IWantToAskAnAmerican, HonestHotTakes, foodquestions, TechNook, randomquestions, NameThisThing
4            50        Tools, CatastrophicFailure, IndieDev, Justrolledintotheshop, gamedev, UkrainianConflict
5            46                GTA6, moviecritic, GamingLeaksAndRumours

In [10]:
# Does LSH change the result? Same graph built from all exact pairs with J >= THRESHOLD
exact_similar = {(a, b): J[a, b] for a, b in zip(i_up, j_up) if J[a, b] >= THRESHOLD}
exact_levels = hp.louvain(hp.similarity_graph(exact_similar, names))
exact_membership, exact_q = exact_levels[-1]
same_pair = [(membership[x] == membership[y]) == (exact_membership[x] == exact_membership[y])
             for x, y in combinations(sorted(set(membership) & set(exact_membership)), 2)]
print(f"exact graph: {len(exact_similar)} edges, {len(set(exact_membership.values()))} communities, Q = {exact_q:.3f}")
print(f"pairs of subreddits grouped the same way in both partitions: {np.mean(same_pair):.3f}")

exact graph: 2919 edges, 40 communities, Q = 0.762


pairs of subreddits grouped the same way in both partitions: 0.962


## 5. Evaluation: null model, vertex order and hold-out recommendation

In [11]:
null = hp.null_model_test(adj, n_null=20)
order = hp.order_stability(adj, n_orders=10)
print(f"unweighted Q real = {null['real']:.3f} | null = {null['null_mean']:.3f} ± {null['null_std']:.3f} | z = {null['z']:.1f}")
print(order)

unweighted Q real = 0.736 | null = 0.365 ± 0.003 | z = 109.4
{'q_mean': 0.7725535530067222, 'q_min': 0.7712503162761001, 'q_max': 0.7739915737188475, 'n_comm_min': 45, 'n_comm_max': 48}


In [12]:
# Hold-out test (item-based collaborative filtering): hide one subreddit per user, rebuild the graph
# without it, and rank all other subreddits by their summed similarity to the user's other subreddits.
# Repeated with five random splits to see how much the result varies.
KS = np.arange(1, 21)
runs = {"LSH graph": [], "exact graph": [], "popularity": []}
for split_seed in range(5):
    split = hp.hold_out(sets, min_memberships=3, seed=split_seed)
    train_sets = [set(split["user_idx"][split["set_idx"] == s]) for s in range(n_sets)]
    train_sig = hp.minhash(split["set_idx"], split["user_idx"], n_sets, k=N_HASH)
    train_similar = hp.verify(hp.lsh_pairs(train_sig, BANDS, ROWS), train_sets, THRESHOLD)
    J_train = hp.jaccard_matrix(split["set_idx"], split["user_idx"], n_sets)
    np.fill_diagonal(J_train, 0)
    recommenders = {"LSH graph": hp.similarity_matrix(train_similar, n_sets),
                    "exact graph": np.where(J_train >= THRESHOLD, J_train, 0),
                    "popularity": np.zeros((n_sets, n_sets))}
    for name, W in recommenders.items():
        runs[name].append(hp.hit_rate(hp.recommendation_ranks(split, W, n_sets, n_users), KS))
hits = {name: np.mean(r, axis=0) for name, r in runs.items()}
hits_se = {name: np.std(r, axis=0, ddof=1) / np.sqrt(len(r)) for name, r in runs.items()}
hits["random"] = KS / (n_sets - 2)
print(f"test users per split: {len(split['test_user'])}")
print(pd.DataFrame({name: [f"{100 * hits[name][k - 1]:.1f} ± {100 * hits_se[name][k - 1]:.1f}" for k in (1, 5, 10, 20)]
                    for name in runs}, index=[1, 5, 10, 20]).to_string())

test users per split: 1853
     LSH graph exact graph  popularity
1    5.3 ± 0.2   5.3 ± 0.2   2.6 ± 0.1
5   14.4 ± 0.3  14.8 ± 0.3   7.4 ± 0.1
10  20.8 ± 0.3  21.2 ± 0.3  10.9 ± 0.2
20  29.9 ± 0.4  30.2 ± 0.5  16.9 ± 0.3


In [13]:
# Figure 4: (a) modularity of the real graph vs. random graphs, (b) hold-out recommendation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(COL * 2.05, 2.0), gridspec_kw={"width_ratios": [1, 1.25]})
jitter = np.random.default_rng(0).uniform(-0.25, 0.25, len(null["null_scores"]))
ax1.scatter(null["null_scores"], jitter, s=14, color=GREY, edgecolors="white", linewidths=0.4,
            label="20 random graphs\n(same degrees)")
ax1.scatter([null["real"]], [0], s=40, marker="D", color=SERIES[0], edgecolors="white", linewidths=0.5,
            label="our graph", zorder=3)
ax1.axvline(0.3, color=INK2, lw=0.8)
ax1.text(0.305, 0.62, "rule of thumb\n$Q>0.3$", color=INK2, fontsize=6.5, va="top")
ax1.set(xlabel="modularity $Q$ (unweighted)", xlim=(0.25, max(0.72, null["real"] + 0.05)), ylim=(-0.7, 0.7), yticks=[])
ax1.grid(axis="y", visible=False)
ax1.spines["left"].set_visible(False)
ax1.legend(loc="lower right", handletextpad=0.3)
ax1.set_title("(a) Null model", loc="left")
for name, color in [("LSH graph", SERIES[0]), ("exact graph", SERIES[1]), ("popularity", SERIES[2]), ("random", GREY)]:
    ax2.plot(KS, 100 * hits[name], color=color, label=name, ls="-" if name != "exact graph" else (0, (1, 1.5)))
ax2.set(xlabel="number of recommendations $k$", ylabel="hidden subreddit found [%]", xlim=(1, 20), ylim=(0, None),
        xticks=[1, 5, 10, 15, 20])
ax2.legend(loc="upper left")
ax2.set_title("(b) Recommending a hidden subreddit", loc="left")
fig.tight_layout(w_pad=1.5)
fig.savefig(FIGURES / "fig_validation.pdf")

## 6. Community labels: TF.IDF (Week 1) vs. log-odds with informative Dirichlet prior (outside method)

In [14]:
docs = hp.community_documents(comments, sub_comm, n_comm)
Y, words = hp.community_term_matrix(docs, min_count=20)
Z = hp.log_odds_z(Y)
labels = pd.DataFrame({
    "size": community_table["size"],
    "log-odds": [", ".join(hp.top_terms(Z, words, c, 6)) for c in range(n_comm)],
    "tf.idf": [", ".join(hp.tfidf_top_terms(Y, words, c, 6)) for c in range(n_comm)],
})
labels["spread log-odds"] = [hp.label_spread(hp.top_terms(Z, words, c, 6), members_of[c], comments) for c in range(n_comm)]
labels["spread tf.idf"] = [hp.label_spread(hp.tfidf_top_terms(Y, words, c, 6), members_of[c], comments) for c in range(n_comm)]
labels["generic log-odds"] = [hp.generic_share(hp.top_terms(Z, words, c, 6), Y, words) for c in range(n_comm)]
labels["generic tf.idf"] = [hp.generic_share(hp.tfidf_top_terms(Y, words, c, 6), Y, words) for c in range(n_comm)]
big = labels[labels["size"] >= 5]
print("vocabulary:", len(words), "terms")
print(labels.round(2).to_string())
print("communities with >= 5 subreddits:", big.iloc[:, 3:].mean().round(2).to_dict())

vocabulary: 5542 terms
           size                                                         log-odds                                                           tf.idf  spread log-odds  spread tf.idf  generic log-odds  generic tf.idf
community                                                                                                                                                                                                          
0            94                              trump, gif, giphy, iran, war, china                               gif, giphy, trump, iran, war, like             0.21           0.35              0.00            0.17
1            83                         gif, giphy, canada, potash, bond, people                          gif, giphy, potash, like, trump, people             0.23           0.40              0.17            0.33
2            78                        model, agent, models, agents, claude, llm                         model, agent, hermes, as

## 7. The community map (Figure 2)

The names are our own short descriptions, based on the subreddits and labels printed above.
Check them after every new data download.

In [15]:
# our names for the largest communities, from their subreddits and labels (check after new data)
COMMUNITY_NAMES = {0: "News and politics", 1: "Viral and general", 2: "AI and agents", 3: "Questions and opinions",
                   4: "Making and fixing", 5: "Games and films", 6: "Software and data", 7: "India",
                   8: "Memes and Linux", 9: "Jobs and careers", 10: "Sports", 11: "German-speaking",
                   12: "California and expats", 13: "United Kingdom", 14: "Academia"}
names_shown = {c: COMMUNITY_NAMES.get(c, " / ".join(members_of[c][:2])) for c in range(N_COLOURED)}

G = nx.Graph()
for (a, c), w in similar.items():
    G.add_edge(names[a], names[c], weight=w)
largest = max(nx.connected_components(G), key=len)
n_components = nx.number_connected_components(G)
G = G.subgraph(largest).copy()
pos = nx.forceatlas2_layout(G, weight="weight", linlog=True, scaling_ratio=2.0, gravity=1.0, max_iter=2000, seed=3)

fig = plt.figure(figsize=(COL * 2.05, 3.3))
ax = fig.add_axes([0, 0, 0.64, 1])
ax.set_axis_off()
nx.draw_networkx_edges(G, pos, ax=ax, width=0.25, edge_color="#c9c8c3", alpha=0.5)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[4 + 0.5 * size[v] for v in G.nodes],
                       node_color=[SERIES[sub_comm[v]] if sub_comm[v] < N_COLOURED else GREY for v in G.nodes],
                       linewidths=0.4, edgecolors="white")
centres = {c: np.mean([pos[v] for v in G.nodes if sub_comm[v] == c], axis=0)
           for c in range(N_COLOURED) if any(sub_comm[v] == c for v in G.nodes)}
span = np.ptp(np.array(list(pos.values())), axis=0)
for _ in range(50):                      # push number labels apart if they overlap
    for a, b in combinations(list(centres), 2):
        gap = (centres[a] - centres[b]) / span
        if np.hypot(*gap) < 0.06:
            step = 0.01 * span * (gap / (np.hypot(*gap) + 1e-9) if np.hypot(*gap) > 0 else np.array([0, 1]))
            centres[a], centres[b] = centres[a] + step, centres[b] - step
for c, xy in centres.items():
    ax.text(*xy, str(c + 1), fontsize=8, fontweight="bold", ha="center", va="center", color=INK,
            bbox=dict(boxstyle="circle,pad=0.15", fc="white", ec=SERIES[c], lw=1.0))
key = fig.add_axes([0.65, 0.02, 0.35, 0.96])
key.set_axis_off()
key.set_xlim(0, 1)
key.set_ylim(0, 1)
for c in range(N_COLOURED):
    y = 0.96 - c * 0.118
    key.scatter([0.04], [y - 0.025], s=55, color=SERIES[c], edgecolors="white", linewidths=0.4)
    key.text(0.04, y - 0.025, str(c + 1), fontsize=6, ha="center", va="center", color="white", fontweight="bold")
    key.text(0.11, y, f"{names_shown[c]} ({len(members_of[c])})", fontsize=7.5, color=INK, va="center")
    key.text(0.11, y - 0.05, ", ".join("r/" + v for v in members_of[c][:3]), fontsize=6, color=INK2, va="center")
key.text(0.04, 0.96 - N_COLOURED * 0.118, f"grey: {n_comm - N_COLOURED} smaller communities", fontsize=6.5, color=INK2, va="center")
fig.savefig(FIGURES / "fig_map.pdf")
print("largest component:", G.number_of_nodes(), "of", len(adj), "subreddits;", n_components, "components")

/usr/local/lib/python3.13/site-packages/networkx/drawing/layout.py:1817: RuntimeWarning: invalid value encountered in divide
  attraction = -np.log(1 + distance) / distance


largest component: 759 of 831 subreddits; 27 components


## 8. Who goes where? Users of the four seed subreddits compared (Figure 5)

Users from exactly one seed subreddit form a group. Lift (Week 5) = share of the group's users who
are members of a community / share among all users. Bootstrap over users gives a 95% interval.

In [16]:
SHOWN = list(range(len(COMMUNITY_NAMES)))
who = hp.seed_lift(activity, sub_comm, SEEDS, SHOWN, min_count=MIN_COUNT)
print("group sizes:", who["sizes"])
print(who["lift"].rename(index=COMMUNITY_NAMES).round(2).to_string())

from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
diverging = LinearSegmentedColormap.from_list("blue_grey_red", [SERIES[0], "#f0efec", SERIES[7]])
log_lift = np.log2(who["lift"].clip(lower=1 / 8).to_numpy())
reliable = (who["low"] > 1) | (who["high"] < 1)               # 95% interval excludes 1
fig, ax = plt.subplots(figsize=(COL, 3.6))
image = ax.imshow(log_lift, cmap=diverging, norm=TwoSlopeNorm(0, -2, 2), aspect="auto")
for (r, c), value in np.ndenumerate(who["lift"].to_numpy()):
    ax.text(c, r, f"{value:.2f}", ha="center", va="center", fontsize=6.5,
            color=INK if reliable.iloc[r, c] else INK2, fontweight="bold" if reliable.iloc[r, c] else "normal")
short = {"datascience": "r/data-\nscience", "MachineLearning": "r/Machine-\nLearning",
         "dataengineering": "r/data-\nengineering", "statistics": "r/statistics\n"}
ax.set_xticks(range(len(SEEDS)), [f"{short.get(s, 'r/' + s)}\n{who['sizes'][s]} users" for s in SEEDS], fontsize=6.5)
ax.set_yticks(range(len(SHOWN)), [COMMUNITY_NAMES[c] for c in SHOWN])
ax.tick_params(length=0)
ax.grid(False)
for side in ax.spines.values():
    side.set_visible(False)
bar = fig.colorbar(image, ax=ax, ticks=[-2, -1, 0, 1, 2], fraction=0.05, pad=0.02)
bar.ax.set_yticklabels(["1/4", "1/2", "1", "2", "4"], fontsize=6.5)
bar.set_label("lift (log scale)", fontsize=7)
bar.outline.set_visible(False)
fig.savefig(FIGURES / "fig_who.pdf")

group sizes: {'datascience': 275, 'MachineLearning': 820, 'dataengineering': 866, 'statistics': 210}
                        datascience  MachineLearning  dataengineering  statistics
News and politics              1.16             0.81             1.07        1.10
Viral and general              1.15             0.92             0.98        1.05
AI and agents                  0.81             1.24             0.76        1.02
Questions and opinions         1.24             0.79             1.02        1.35
Making and fixing              1.10             1.04             0.99        0.66
Games and films                0.99             1.06             0.88        0.98
Software and data              1.07             0.34             1.60        0.44
India                          0.64             1.16             1.12        0.60
Memes and Linux                0.86             0.94             1.04        0.68
Jobs and careers               1.29             0.68             1.22        0.

## 9. Export numbers and tables for the report (`report/results.tex`)

In [17]:
def latex(text):
    return str(text).replace("_", r"\_").replace("&", r"\&")


def latex_list(items, n=5):
    return ", ".join(latex(x) for x in items[:n])


numbers = {
    "NUSERS": f"{activity['user'].nunique():,}", "NROWS": f"{len(activity):,}", "NSUBSALL": f"{activity['subreddit'].nunique():,}",
    "MEDIANSUBS": int(activity.groupby("user").size().median()), "MINCOUNT": MIN_COUNT, "MINUSERS": MIN_USERS,
    "NSUBS": f"{n_sets:,}", "NPAIRS": f"{n_sets * (n_sets - 1) // 2:,}", "NMEMBERS": f"{len(sets['set_idx']):,}",
    "NSETUSERS": f"{n_users:,}", "SETMEDIAN": int(np.median(sets["sizes"])), "SETMAX": int(np.max(sets["sizes"])),
    "SHAREZERO": f"{100 * (upper == 0).mean():.0f}", "JQNINETYNINE": f"{np.quantile(upper, 0.99):.2f}",
    "SHARESIMILAR": f"{100 * (upper >= THRESHOLD).mean():.1f}",
    "NHASH": N_HASH, "BANDS": BANDS, "ROWS": ROWS, "THRESHOLD": THRESHOLD,
    "LSHT": f"{(1 / BANDS) ** (1 / ROWS):.2f}", "NSIMILAR": f"{quality['similar']:,}", "NCAND": f"{quality['candidates']:,}",
    "CANDSHARE": f"{100 * quality['candidates'] / quality['pairs']:.1f}", "RECALL": f"{100 * quality['recall']:.0f}",
    "NFN": f"{quality['false_neg']:,}", "NFP": f"{quality['false_pos']:,}",
    "RECALLLECTURE": f"{100 * band_choices.set_index(['n', 'b', 'r']).loc[(100, 20, 5), 'recall']:.0f}",
    "RECALLHUNDRED": f"{100 * band_choices.set_index(['n', 'b', 'r']).loc[(100, 50, 2), 'recall']:.0f}",
    "CHECKONE": f"{100 * band_choices.set_index(['n', 'b', 'r']).loc[(N_HASH, N_HASH, 1), 'candidates'] / quality['pairs']:.0f}",
    "TIMENAIVE": f"{time_naive:.2f}", "TIMELSH": f"{time_lsh:.2f}",
    "SPEEDUP": f"{time_naive / time_lsh:.0f}",
    "NEDGES": f"{len(similar):,}", "NGRAPH": f"{len(adj):,}", "NCOMM": n_comm, "QW": f"{q_final:.2f}",
    "NLCC": f"{G.number_of_nodes():,}", "NCOMPONENTS": n_components,
    "NEDGESEXACT": f"{len(exact_similar):,}", "NCOMMEXACT": len(set(exact_membership.values())), "QEXACT": f"{exact_q:.2f}",
    "SAMEPAIR": f"{100 * np.mean(same_pair):.0f}",
    "QREAL": f"{null['real']:.2f}", "QNULL": f"{null['null_mean']:.2f}", "QNULLSTD": f"{null['null_std']:.3f}",
    "ZNULL": f"{null['z']:.0f}", "QMIN": f"{order['q_min']:.2f}", "QMAX": f"{order['q_max']:.2f}",
    "NCOMMMIN": order["n_comm_min"], "NCOMMMAX": order["n_comm_max"],
    "QSPREAD": f"{order['q_max'] - order['q_min']:.3f}",
    "NTEST": f"{len(split['test_user']):,}",
    "NSPLITS": len(runs["LSH graph"]),
    "HITLSH": f"{100 * hits['LSH graph'][9]:.0f}", "HITEXACT": f"{100 * hits['exact graph'][9]:.0f}",
    "HITLSHSE": f"{100 * hits_se['LSH graph'][9]:.1f}", "HITPOPSE": f"{100 * hits_se['popularity'][9]:.1f}",
    "HITPOP": f"{100 * hits['popularity'][9]:.0f}", "HITRAND": f"{100 * hits['random'][9]:.0f}",
    "HITONE": f"{100 * hits['LSH graph'][0]:.0f}", "HITONEPOP": f"{100 * hits['popularity'][0]:.0f}",
    "NCOMMENTS": f"{len(comments):,}", "NVOCAB": f"{len(words):,}", "NCOMMBIG": len(big),
    "NCOMMSUBS": f"{comments.loc[comments['subreddit'].isin(names), 'subreddit'].nunique():,}",
    "SPREADLO": f"{100 * big['spread log-odds'].mean():.0f}", "SPREADTF": f"{100 * big['spread tf.idf'].mean():.0f}",
    "GENERICLO": f"{100 * big['generic log-odds'].mean():.0f}", "GENERICTF": f"{100 * big['generic tf.idf'].mean():.0f}",
}
name_of = {v: k for k, v in COMMUNITY_NAMES.items()}
def lift_of(community, seed):
    return f"{who['lift'].loc[name_of[community], seed]:.2f}"
numbers.update({
    "NSEEDGROUPS": f"{sum(who['sizes'].values()):,}",
    "LIFTAIML": lift_of("AI and agents", "MachineLearning"),
    "LIFTSOFTDE": lift_of("Software and data", "dataengineering"),
    "LIFTSOFTML": lift_of("Software and data", "MachineLearning"),
    "LIFTACADML": lift_of("Academia", "MachineLearning"),
    "LIFTACADST": lift_of("Academia", "statistics"),
    "LIFTACADDE": lift_of("Academia", "dataengineering"),
    "LIFTJOBSDS": lift_of("Jobs and careers", "datascience"),
    "LIFTJOBSDE": lift_of("Jobs and careers", "dataengineering"),
})
lines = [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in numbers.items()]
lines += [f"\\newcommand{{\\COMMNAME{chr(65 + c)}}}{{{latex(names_shown[c])}}}" for c in range(N_COLOURED)]

band_rows = []
for row in band_choices.itertuples():
    cells = [str(row.n), str(row.b), str(row.r), f"{row.threshold:.2g}", f"{100 * row.candidates / row.pairs:.1f}\\%",
             f"{100 * row.recall:.0f}\\%"]
    if (row.n, row.b, row.r) == (N_HASH, BANDS, ROWS):
        cells = [f"\\textbf{{{c}}}" for c in cells]
    band_rows.append(" & ".join(cells) + " \\\\")
lines.append("\\newcommand{\\BANDTABLE}{" + "\n".join(band_rows) + "}")

shown = labels.iloc[:N_COLOURED]
comm_rows = [f"{c + 1} & {latex(names_shown[c])} & {row.size} & {latex_list(row._2.split(', '), 3)} & "
             f"{latex_list(row._3.split(', '), 3)} \\\\" for c, row in zip(shown.index, shown.itertuples())]
lines.append("\\newcommand{\\COMMTABLE}{" + "\n".join(comm_rows) + "}")
Path("report/results.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines[:15]))

\newcommand{\NUSERS}{2,999}
\newcommand{\NROWS}{91,836}
\newcommand{\NSUBSALL}{19,626}
\newcommand{\MEDIANSUBS}{19}
\newcommand{\MINCOUNT}{2}
\newcommand{\MINUSERS}{10}
\newcommand{\NSUBS}{956}
\newcommand{\NPAIRS}{456,490}
\newcommand{\NMEMBERS}{27,411}
\newcommand{\NSETUSERS}{2,470}
\newcommand{\SETMEDIAN}{17}
\newcommand{\SETMAX}{505}
\newcommand{\SHAREZERO}{67}
\newcommand{\JQNINETYNINE}{0.08}
\newcommand{\SHARESIMILAR}{0.6}
